# Week 5 — Language Modeling: From n-grams to Neural

We formalize the language modeling task, define perplexity, and trace the architectural evolution from count-based n-grams to the feed-forward neural language model of Bengio et al. (2003) — the architecture that introduced the embedding layer and the softmax output, and made everything that follows in this course possible.

## Learning Objectives

- Define a language model formally and connect it to the chain rule of probability.
- Derive perplexity from cross-entropy and explain its interpretive scale.
- Implement an n-gram LM with Kneser–Ney smoothing.
- Implement the Bengio (2003) neural LM in PyTorch and reproduce its perplexity advantage.

## Required Reading

- Bengio, Y., Ducharme, R., Vincent, P., & Jauvin, C. (2003). *A Neural Probabilistic Language Model*.
- Jurafsky & Martin, Chapters 3 and 7.

In [ ]:
import sys, math, re
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

np.random.seed(0)

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    HAS_TORCH = True
    torch.manual_seed(0)
except ImportError:
    HAS_TORCH = False
    print("PyTorch not available — neural LM section will be skipped.")

## 1. The language modeling task

A language model assigns a probability $P(w_1, \ldots, w_T)$ to any token sequence. Via the chain rule,

$$P(w_1, \ldots, w_T) = \prod_{t=1}^{T} P(w_t \mid w_{<t}).$$

The modeling problem is to estimate $P(w_t \mid w_{<t})$. Everything in modern NLP — translation, summarization, dialogue, code generation — is downstream of this single problem.

## 2. Perplexity

The cross-entropy of a model $q$ on text drawn from $p$ is

$$H(p, q) = -\mathbb{E}_{w \sim p} \log q(w).$$

Perplexity is its exponential:

$$\text{PPL}(q) = \exp(H(p, q)) = \exp\!\left(-\frac{1}{N} \sum_{t=1}^{N} \log q(w_t \mid w_{<t})\right).$$

Interpretation: a perplexity of $K$ means the model is, on average, as uncertain as if choosing uniformly among $K$ alternatives. A uniform model over a vocabulary of size $V$ has $\text{PPL} = V$; a perfect model has $\text{PPL} = 1$.

**Warning.** Perplexity is *not* directly comparable across tokenizers. Per-byte perplexity (bits per byte) is the right cross-tokenizer measure.

## 3. N-gram language models with Kneser–Ney smoothing

A trigram MLE is

$$P_{\text{MLE}}(w \mid u, v) = \frac{c(u, v, w)}{c(u, v)}.$$

Zero counts assign zero probability, so we smooth. **Modified Kneser–Ney** (Chen & Goodman, 1999) is the empirically strongest n-gram smoothing method:

$$P_{\text{KN}}(w \mid u, v) = \frac{\max(c(u, v, w) - D, 0)}{c(u, v)} + \lambda(u, v) P_{\text{cont}}(w \mid v),$$

where the continuation probability $P_{\text{cont}}$ weights words by the *number of distinct contexts they appear in*, not their raw frequency — the Kneser–Ney insight that *San Francisco* is frequent but rarely a novel continuation.

In [ ]:
class TrigramLM:
    def __init__(self, discount=0.75):
        self.D = discount

    def fit(self, sentences):
        self.tri = Counter()
        self.bi = Counter()
        self.uni = Counter()
        self.bi_ctx = Counter()    # count(u, v) for trigram contexts
        self.uni_ctx = Counter()   # count(v) for bigram contexts
        self.follow_3 = defaultdict(set)  # words that follow (u, v)
        self.follow_2 = defaultdict(set)  # words that follow v
        self.precede_2 = defaultdict(set) # (u, v) pairs preceded by v ... no, preceding distinct types
        self.precede_1 = defaultdict(set) # distinct words preceding w (for continuation prob)
        self.distinct_bigrams = 0

        for sent in sentences:
            s = ['<s>', '<s>'] + sent + ['</s>']
            for i in range(len(s)):
                self.uni[s[i]] += 1
                if i >= 1:
                    self.bi[(s[i-1], s[i])] += 1
                    self.uni_ctx[s[i-1]] += 1
                    self.follow_2[s[i-1]].add(s[i])
                    self.precede_1[s[i]].add(s[i-1])
                if i >= 2:
                    self.tri[(s[i-2], s[i-1], s[i])] += 1
                    self.bi_ctx[(s[i-2], s[i-1])] += 1
                    self.follow_3[(s[i-2], s[i-1])].add(s[i])

        self.distinct_bigrams = len(self.bi)
        self.V = set(self.uni.keys())
        return self

    def p_continuation(self, w):
        # P_cont(w) ∝ |{u : c(u, w) > 0}|
        num = len(self.precede_1.get(w, set()))
        return num / max(self.distinct_bigrams, 1)

    def p_kn_bigram(self, w, v):
        c_vw = self.bi[(v, w)]
        c_v = self.uni_ctx[v]
        if c_v == 0:
            return self.p_continuation(w)
        lam = (self.D * len(self.follow_2[v])) / c_v
        return max(c_vw - self.D, 0) / c_v + lam * self.p_continuation(w)

    def p_kn_trigram(self, w, u, v):
        c_uvw = self.tri[(u, v, w)]
        c_uv = self.bi_ctx[(u, v)]
        if c_uv == 0:
            return self.p_kn_bigram(w, v)
        lam = (self.D * len(self.follow_3[(u, v)])) / c_uv
        return max(c_uvw - self.D, 0) / c_uv + lam * self.p_kn_bigram(w, v)

    def perplexity(self, sentences):
        log_p, n = 0.0, 0
        for sent in sentences:
            s = ['<s>', '<s>'] + sent + ['</s>']
            for i in range(2, len(s)):
                p = max(self.p_kn_trigram(s[i], s[i-2], s[i-1]), 1e-12)
                log_p += math.log(p)
                n += 1
        return math.exp(-log_p / max(n, 1))

def tokenize(s):
    return re.findall(r"\w+", s.lower())

# A modest training corpus (in production: Penn Treebank, WikiText, etc.)
TRAIN_TEXT = (
    "the cat sat on the mat. the dog sat on the rug. "
    "machine learning models learn from data. deep learning is powerful. "
    "natural language processing handles text. the quick brown fox jumps. "
    "language models predict words. neural networks have many parameters. "
    "transformers process sequences in parallel. attention is all you need. " * 60
)
TEST_TEXT = (
    "the cat chased the mouse. the dog barked at the cat. "
    "neural networks predict words from context. "
)

train_sents = [tokenize(s) for s in TRAIN_TEXT.split('.') if s.strip()]
test_sents  = [tokenize(s) for s in TEST_TEXT.split('.') if s.strip()]

lm = TrigramLM(discount=0.75).fit(train_sents)
print(f"Train perplexity (KN trigram): {lm.perplexity(train_sents):.2f}")
print(f"Test  perplexity (KN trigram): {lm.perplexity(test_sents):.2f}")

## 4. The Bengio (2003) neural language model

The architecture: feed-forward net mapping the last $n - 1$ tokens to a softmax over the vocabulary.

$$\mathbf{h} = \tanh(W [\mathbf{e}_{w_{t-n+1}}; \ldots; \mathbf{e}_{w_{t-1}}] + \mathbf{b}),$$

$$P(w_t \mid w_{<t}) = \text{softmax}(U \mathbf{h} + \mathbf{d}).$$

Three innovations from this paper that we still use:

1. **Learned word embeddings.** $\mathbf{e}_w \in \mathbb{R}^m$ — a dense vector per word, learned jointly with the LM.
2. **Softmax output.** Probability over the full vocabulary.
3. **End-to-end gradient training.** All parameters learned by SGD on log-likelihood.

In [ ]:
if HAS_TORCH:
    class BengioLM(nn.Module):
        def __init__(self, vocab_size, emb_dim=32, hidden=64, context=3):
            super().__init__()
            self.context = context
            self.emb = nn.Embedding(vocab_size, emb_dim)
            self.W = nn.Linear(context * emb_dim, hidden)
            self.U = nn.Linear(hidden, vocab_size)

        def forward(self, idx):
            # idx: (batch, context)
            e = self.emb(idx).reshape(idx.size(0), -1)
            h = torch.tanh(self.W(e))
            return self.U(h)  # logits (batch, vocab)

    # Build vocab from training sentences
    all_tokens = ['<s>', '</s>'] + sorted({t for s in train_sents for t in s})
    tok2id = {t: i for i, t in enumerate(all_tokens)}
    id2tok = {i: t for t, i in tok2id.items()}
    V = len(tok2id)
    CONTEXT = 3

    def make_examples(sents):
        X, Y = [], []
        for s in sents:
            ids = [tok2id['<s>']] * CONTEXT + [tok2id[t] for t in s if t in tok2id] + [tok2id['</s>']]
            for i in range(CONTEXT, len(ids)):
                X.append(ids[i-CONTEXT:i])
                Y.append(ids[i])
        return torch.tensor(X), torch.tensor(Y)

    X_train, Y_train = make_examples(train_sents)
    X_test,  Y_test  = make_examples([s for s in test_sents if all(t in tok2id for t in s)])

    model = BengioLM(V, emb_dim=32, hidden=64, context=CONTEXT)
    opt = torch.optim.Adam(model.parameters(), lr=1e-2)

    for epoch in range(200):
        model.train()
        opt.zero_grad()
        logits = model(X_train)
        loss = F.cross_entropy(logits, Y_train)
        loss.backward(); opt.step()
        if (epoch + 1) % 50 == 0:
            with torch.no_grad():
                test_loss = F.cross_entropy(model(X_test), Y_test).item() if len(X_test) > 0 else float('nan')
            print(f"epoch {epoch+1:4d}  train_loss={loss.item():.4f}  test_loss={test_loss:.4f}")

    # Final perplexities
    with torch.no_grad():
        train_ppl = math.exp(F.cross_entropy(model(X_train), Y_train).item())
        test_ppl  = math.exp(F.cross_entropy(model(X_test),  Y_test ).item()) if len(X_test) > 0 else float('nan')
    print(f"\nBengio LM — train PPL: {train_ppl:.2f},  test PPL: {test_ppl:.2f}")
    print(f"Trigram KN    — train PPL: {lm.perplexity(train_sents):.2f},  test PPL: {lm.perplexity(test_sents):.2f}")
else:
    print("Install PyTorch to run this section.")

## 5. Sampling from a language model

A trained LM can generate text by sampling tokens autoregressively. Three common strategies:

- **Greedy.** Always take the argmax. Repetitive, low-diversity.
- **Temperature sampling.** Scale logits by $1/\tau$ before softmax. $\tau < 1$ → sharper, $\tau > 1$ → flatter.
- **Top-k / top-p (nucleus).** Restrict the support before sampling (Holtzman et al., 2020).

In [ ]:
if HAS_TORCH:
    def sample(model, prompt_ids, max_new=20, temperature=1.0, top_k=None):
        model.eval()
        out = list(prompt_ids)
        for _ in range(max_new):
            ctx = torch.tensor([out[-CONTEXT:]])
            with torch.no_grad():
                logits = model(ctx).squeeze() / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[-1]] = -float('inf')
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, 1).item()
            out.append(next_id)
            if next_id == tok2id['</s>']:
                break
        return out

    seed = [tok2id['<s>']] * CONTEXT + [tok2id.get('the', tok2id['<s>'])]
    for temp in [0.5, 1.0, 1.5]:
        ids = sample(model, seed, max_new=15, temperature=temp)
        words = [id2tok[i] for i in ids[CONTEXT:]]
        print(f"τ={temp}: {' '.join(words)}")
else:
    print("Skipped (no PyTorch).")

## 6. Exercises

1. **Perplexity by hand.** Compute the perplexity of a uniform model over $|V| = 10000$ on any held-out text. Then compute the perplexity of a model that always predicts the unigram distribution. Which is lower, and by how much?
2. **Tokenization invariance.** Show by counterexample that perplexity is *not* invariant under different tokenizations of the same text. State a tokenizer-independent metric.
3. **Hierarchical softmax.** The softmax over a 50 k vocabulary is the bottleneck of the Bengio model. Implement hierarchical softmax (Morin & Bengio, 2005) — a binary tree over the vocabulary — and measure the speedup.
4. **Scaling exercise.** Train the Bengio LM with context lengths $n \in \{2, 3, 5, 10\}$ and hidden sizes $\{32, 128, 512\}$. Plot test perplexity vs. parameter count. Identify the scaling regime.

---

## Next Week

Week 6 — Recurrent architectures. RNN, LSTM, GRU, and BPTT — including a hands-on proof of the vanishing-gradient theorem.